# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [3]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/self-reliance.pdf")

docs = loader.load()

Let's look at an example document to see if everything worked as expected!

In [5]:
docs[0].page_content

'Self-Reliance\nRalph Waldo Emerson\n1841\n\\Ne te quaesiveris extra."\n\\Man is his own star; and the soul that can\nRender an honest and a perfect man,\nCommands all light, all in\ruence, all fate;\nNothing to him falls early or too late.\nOur acts our angels are, or good or ill,\nOur fatal shadows that walk by us still."\nEpilogue to Beaumont and Fletcher\'s Honest Man\'s Fortune\nCast the bantling on the rocks,\nSuckle him with the she-wolf\'s teat;\nWintered with the hawk and fox,\nPower and speed be hands and feet.\nI read the other day some verses written by an eminent painter which\nwere original and not conventional. The soul always hears an admonition\nin such lines, let the subject be what it may. The sentiment they instil is\nof more value than any thought they may contain. To believe your own\nthought, to believe that what is true for you in your private heart is true\nfor all men, | that is genius. Speak your latent conviction, and it shall\nbe the universal sense; for th

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    docs,
    embeddings,
    location=":memory:",
    collection_name="Ralph Waldo Emerson"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

In [8]:
# Size information about naive_retriever
print(f"Number of documents in the vector store: {len(docs)}")
print(f"Number of documents retrieved per query (k): {naive_retriever.search_kwargs['k']}")
print(f"Retriever type: {type(naive_retriever).__name__}")


Number of documents in the vector store: 21
Number of documents retrieved per query (k): 10
Retriever type: VectorStoreRetriever


### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [9]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [10]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [11]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [12]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain based on the provided context appears to be self-reliance, personal development, and individual virtue. The text emphasizes themes such as independence, trusting oneself, inner strength, and the importance of personal virtues over societal expectations.'

In [13]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there do not appear to be any specific use cases about security mentioned in the document.'

In [14]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had nothing to say about the fintech projects in the provided context.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [15]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(docs)

We'll construct the same chain - only changing the retriever.

In [16]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [17]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

"The most common project domain in the provided context appears to be related to personal development and philosophical reflection. The text emphasizes themes such as self-trust, intuition, originality, and individual worth. It discusses concepts like self-reliance, the importance of trusting one's perceptions and instincts, and the value of originality and nonconformity.\n\nSo, the most common project domain suggested by the context is **Self-Development or Personal Philosophy**."

In [18]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'The provided context does not mention any use cases specifically related to security.'

In [19]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had what to say about the fintech projects?'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer 

For keyword lookups where lexical differnce is key


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [20]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [21]:
contextual_compression_retrieval_chain = (
    {
        "context": itemgetter("question") | compression_retriever, 
        "question": itemgetter("question")
        }
    | RunnablePassthrough.assign(context=itemgetter("context"))

    | {
        "response": rag_prompt | chat_model, 
        "context": itemgetter("context")
        }
)

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided context, I do not have specific information about the most common project domain.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there is no specific mention of use cases related to security.'

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The judges had favorable opinions about the fintech projects. They praised the projects for being good and emphasized the importance of integrity and authenticity in the work. They appreciated that the projects demonstrated genuine effort and virtue, rather than superficial or performative actions. The judges valued the sincerity and true character of the projects, recognizing that true greatness lies in living according to one's principles and doing meaningful work."

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [25]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [26]:
multi_query_retrieval_chain = (
    {
        "context": itemgetter("question") | multi_query_retriever, 
        "question": itemgetter("question")
        }
    
    | RunnablePassthrough.assign(context=itemgetter("context"))

    | {
        "response": rag_prompt | chat_model, 
        "context": itemgetter("context")
        }
)

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

"The most common project domain appears to be focused on self-reliance and individual development. The text emphasizes personal independence, trusting oneself, and developing one's innate virtues and talents rather than conforming to societal norms or imitating others. It discusses themes like inner strength, originality, conscience, and living authentically, which suggest that the primary focus is on personal growth and self-improvement."

In [28]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases about security discussed in the context. Specifically, the text mentions the importance of self-trust and reliance on one\'s own inner strength as a form of security. It emphasizes that "nothing can bring you peace but yourself" and that true power and security come from within, through confidence in one\'s own principles and instincts rather than external means or institutions. The concept of standing alone, trusting your own thought, and maintaining independence is highlighted as a way to achieve genuine security.'

In [29]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had to say very little about the fintech projects directly, as the provided context does not include specific comments or opinions from judges regarding these projects. If you are looking for their opinions or evaluations, I do not have that information from the given text.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

When the question is reformualted, it could fetch slightly different embedded documents, all of which might be relevant. 

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [30]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = docs
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [31]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", 
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"), 
    client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [32]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [33]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [34]:
parent_document_retrieval_chain = (
    {
        "context": itemgetter("question") | parent_document_retriever, 
        "question": itemgetter("question")
        }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {
        "response": rag_prompt | chat_model, 
        "context": itemgetter("context")
        }
)

Let's give it a whirl!

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided context, the most common project domain appears to be related to self-reliance, individual virtue, and spiritual or philosophical development. The text emphasizes themes such as independence from external institutions, the importance of inner strength, and staying true to oneself. Therefore, the most common project domain inferred from this content is likely to be **personal growth and self-reliance**.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there is no explicit mention or discussion of security use cases.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The judges had positive and encouraging remarks about the fintech projects. They appreciated the projects' innovation and potential for growth in the financial technology sector. The judges highlighted the importance of these projects in transforming traditional financial services and emphasized the value of originality and practical application. Overall, their comments reflected support and optimism for the future prospects of the fintech initiatives."

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [38]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [39]:
ensemble_retrieval_chain = (
    {
        "context": itemgetter("question") | ensemble_retriever, 
        "question": itemgetter("question")
        }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {
        "response": rag_prompt | chat_model, 
        "context": itemgetter("context")
        }
)

Let's look at our results!

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain appears to be related to self-reliance, individualism, and personal development, as indicated by the context which is excerpts from Ralph Waldo Emerson\'s essay "Self-Reliance." The content emphasizes themes of independence, nonconformity, inner strength, and trust in oneself rather than specific technical or business project domains.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are references to security related to self-trust and independence. Emerson discusses the importance of trusting oneself, acting with integrity, and standing alone against societal pressures. He emphasizes that true strength and security come from inner reliance, self-awareness, and independence, rather than relying on external institutions, customs, or property. This inner self-trust acts as a form of security, enabling individuals to act authentically and with conviction, regardless of societal conformity or opposition.'

In [42]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had positive things to say about the fintech projects. The context indicates that there was recognition of the importance and value of integrity and originality in individual actions and ideas, which can be related to innovation in fintech. The emphasis on self-trust, independence, and the pursuit of principles over conformity suggests that judges appreciated innovative and genuine efforts like fintech projects that demonstrate originality and integrity. However, the provided content primarily focuses on broader philosophical themes of self-reliance, individualism, and societal conformity, and does not include specific comments or judgments about particular fintech projects. Therefore, I do not have specific details on what judges had to say about the fintech projects.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [43]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [44]:
semantic_documents = semantic_chunker.split_documents(docs[:20])

Let's create a new vector store.

In [45]:
# semantic_documents


In [46]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [47]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [48]:
semantic_retrieval_chain = (
    {
        "context": itemgetter("question") | semantic_retriever, 
        "question": itemgetter("question")
        }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {
        "response": rag_prompt | chat_model, 
        "context": itemgetter("context")
        }
)

And view the results!

In [49]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided context, the most common project domain appears to be centered around self-reliance, individual virtue, and personal empowerment. The text discusses themes such as self-trust, individual action, the importance of personal character, and the influence of single individuals on history and society. It emphasizes the value of personal originality and the power of the individual to effect change across various aspects of life, including religion, education, pursuits, and property.\n\nIn summary, the most common project domain suggested by the text is **personal development and self-reliance**.'

In [50]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'The provided context does not mention any specific usecases related to security.'

In [51]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges did not have any specific comments or opinions about the fintech projects in the provided context.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer

One way would be to make the percentile higher. The default is 95%, so maybe it could be set to like 98%. 
Gradient could also be used, which is specialized for highly repetative data. 

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

### YOUR CODE HERE

1. generate dateset
    2. this will have cols like question, reference_context, reference
2. run the model on each question, get the actual context, and the answer
3. evaluate reference vs the answer with like helpfulnes and other metrics

In [52]:
from ragas.testset import TestsetGenerator

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)


/var/folders/36/s6cm4m0959x3z451g1xvcyy40000gn/T/ipykernel_43229/1464795151.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
/var/folders/36/s6cm4m0959x3z451g1xvcyy40000gn/T/ipykernel_43229/1464795151.py:8: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [57]:
dataset = generator.generate_with_langchain_docs(docs, testset_size=20)

Applying HeadlinesExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/21 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/34 [00:00<?, ?it/s]

Property 'summary' already exists in node '6b9e79'. Skipping!
Property 'summary' already exists in node 'd212cc'. Skipping!
Property 'summary' already exists in node '884a5e'. Skipping!
Property 'summary' already exists in node '46e061'. Skipping!
Property 'summary' already exists in node '4ba506'. Skipping!
Property 'summary' already exists in node 'e1cb41'. Skipping!
Property 'summary' already exists in node '3df44a'. Skipping!
Property 'summary' already exists in node 'a67b7e'. Skipping!
Property 'summary' already exists in node '712947'. Skipping!
Property 'summary' already exists in node '999462'. Skipping!
Property 'summary' already exists in node '65462f'. Skipping!
Property 'summary' already exists in node '2eb76c'. Skipping!
Property 'summary' already exists in node '6924a4'. Skipping!
Property 'summary' already exists in node '5223a2'. Skipping!
Property 'summary' already exists in node '595251'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

...

Applying NERExtractor:   0%|          | 0/7 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [58]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How does the concept of the voyage relate to u...,[The voyage of the best ship is a zigzag line ...,The voyage of the best ship is described as a ...,single_hop_specific_query_synthesizer
1,How does the voyage of the best ship illustrat...,[The voyage of the best ship is a zigzag line ...,The voyage of the best ship is described as a ...,single_hop_specific_query_synthesizer
2,Who is Wesley in the context of personal worth?,"[of Wesley; Abolition, of Clarkson. Scipio, Mi...",The context does not provide specific informat...,single_hop_specific_query_synthesizer
3,Why is Rome important in personal development?,"[of Wesley; Abolition, of Clarkson. Scipio, Mi...",The context discusses how history often simpli...,single_hop_specific_query_synthesizer
4,Why kings make people trust themselves?,"[The world has been instructed by its kings, w...","The context explains that kings, as colossal s...",single_hop_specific_query_synthesizer
5,Wha are kings?,"[The world has been instructed by its kings, w...","The world has been instructed by its kings, wh...",single_hop_specific_query_synthesizer
6,What is the nature of the self according to th...,"[Self, on which a universal reliance may be gr...",The context describes the self as the source o...,single_hop_specific_query_synthesizer
7,How does the concept of Self relate to the sou...,"[Self, on which a universal reliance may be gr...",The context describes the Self as the primary ...,single_hop_specific_query_synthesizer
8,What does the context suggest about the nature...,[The relations of the soul to the divine spiri...,The context indicates that the relations of th...,single_hop_specific_query_synthesizer
9,"According to the context, how should one under...",[The relations of the soul to the divine spiri...,The relations of the soul to the divine spirit...,single_hop_specific_query_synthesizer


In [100]:
import time
from copy import deepcopy
from datetime import datetime

from ragas import EvaluationDataset, RunConfig, evaluate
from ragas.metrics import (ContextEntityRecall,
                           LLMContextPrecisionWithoutReference,
                           LLMContextRecall, NoiseSensitivity,
                           ResponseRelevancy)

In [ ]:
retrievers = {}

retrievers['baseline'] = naive_retrieval_chain
retrievers['bm25_retrieval_chain'] = bm25_retrieval_chain
retrievers['contextual_compression_retrieval_chain'] = contextual_compression_retrieval_chain
retrievers['multi_query_retrieval_chain'] = multi_query_retrieval_chain
retrievers['parent_document_retrieval_chain'] = parent_document_retrieval_chain
retrievers['ensemble_retrieval_chain'] = ensemble_retrieval_chain
retrievers['semantic_retrieval_chain'] = semantic_retrieval_chain



In [106]:
all_results = {}


for retreiver_chain_name, retreiver_chain in retrievers.items():
    print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print(f'testing {retreiver_chain_name}')
    
    dataset_for_this_retriever = deepcopy(dataset)

    x = 0
    for test_row in dataset_for_this_retriever:
        response = retreiver_chain.invoke({"question" : test_row.eval_sample.user_input})
        test_row.eval_sample.response = response["response"].content
        test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
        time.sleep(10) # To try to avoid rate limiting.
        print(x); x += 1


    evaluation_dataset = EvaluationDataset.from_pandas(dataset_for_this_retriever.to_pandas())

    print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print('running the evaluation now ...')
    evaluation_result = evaluate(
        dataset=evaluation_dataset,
        metrics=[LLMContextPrecisionWithoutReference(), LLMContextRecall(), ContextEntityRecall(), NoiseSensitivity(), ResponseRelevancy()],
        llm=LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini")),
        run_config=RunConfig(timeout=360)
    )

    all_results[retreiver_chain_name] = evaluation_result

    print()
    print()



2025-10-12 08:13:30
testing baseline
0
1
2
3
4
5
6
7
8
9
10
11
2025-10-12 08:15:57
running the evaluation now ...


/var/folders/36/s6cm4m0959x3z451g1xvcyy40000gn/T/ipykernel_43229/2119575805.py:26: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  llm=LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini")),


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.




2025-10-12 08:23:00
testing bm25_retrieval_chain
0
1
2
3
4
5
6
7
8
9
10
11
2025-10-12 08:25:23
running the evaluation now ...


...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[28]: TimeoutError()
Exception raised in Job[33]: TimeoutError()
Exception raised in Job[38]: TimeoutError()
Exception raised in Job[43]: TimeoutError()
Exception raised in Job[53]: TimeoutError()
Exception raised in Job[58]: TimeoutError()




2025-10-12 09:01:13
testing semantic_retrieval_chain
0
1
2
3
4
5
6
7
8
9
10
11
2025-10-12 09:03:39
running the evaluation now ...


/var/folders/36/s6cm4m0959x3z451g1xvcyy40000gn/T/ipykernel_43229/2119575805.py:26: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  llm=LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini")),


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[43]: TimeoutError()
Exception raised in Job[53]: TimeoutError()
Exception raised in Job[58]: TimeoutError()


In [157]:
all_results

{'baseline': {'llm_context_precision_without_reference': 0.8713, 'context_recall': 1.0000, 'context_entity_recall': 0.2348, 'noise_sensitivity(mode=relevant)': 0.4527, 'answer_relevancy': 0.9338},
 'bm25_retrieval_chain': {'llm_context_precision_without_reference': 0.8657, 'context_recall': 0.9167, 'context_entity_recall': 0.2302, 'noise_sensitivity(mode=relevant)': 0.2813, 'answer_relevancy': 0.9562},
 'contextual_compression_retrieval_chain': {'llm_context_precision_without_reference': 0.9861, 'context_recall': 0.9722, 'context_entity_recall': 0.2547, 'noise_sensitivity(mode=relevant)': 0.3070, 'answer_relevancy': 0.9492},
 'multi_query_retrieval_chain': {'llm_context_precision_without_reference': 0.8631, 'context_recall': 0.9722, 'context_entity_recall': 0.2302, 'noise_sensitivity(mode=relevant)': 0.5166, 'answer_relevancy': 0.8630},
 'parent_document_retrieval_chain': {'llm_context_precision_without_reference': 0.9931, 'context_recall': 0.9167, 'context_entity_recall': 0.2351, 'noi

In [159]:
import pandas

all_scores_as_df = pandas.DataFrame()

for retriever_name, evaluation_result in all_results.items():

    evalutation_result__all_scores = evaluation_result.scores
    evalutation_result__mean = pandas.DataFrame(evalutation_result__all_scores).mean()

    all_scores_as_df[retriever_name] = evalutation_result__mean

        
all_scores_as_df = all_scores_as_df


In [160]:
all_scores_as_df

,baseline,bm25_retrieval_chain,contextual_compression_retrieval_chain,multi_query_retrieval_chain,parent_document_retrieval_chain,ensemble_retrieval_chain,semantic_retrieval_chain
llm_context_precision_without_reference,0.871291,0.865741,0.986111,0.863096,0.993056,0.924647,0.888606
context_recall,1.000000,0.916667,0.972222,0.972222,0.916667,1.000000,0.986111
context_entity_recall,0.234804,0.230174,0.254684,0.230174,0.235076,0.230174,0.274619
noise_sensitivity(mode=relevant),0.452747,0.281339,0.307027,0.516642,0.363021,0.515171,0.395645
answer_relevancy,0.933834,0.956237,0.949158,0.862983,0.956234,0.945242,0.931095
